# MR4010 Navegacion Autonoma — BC Evasion Model (dataset_mode_v5)

### Reentrenamiento de bc_evasion_model.pt con dataset capturado en el nuevo mundo

Este notebook entrena la red DAVE-2 sobre el dataset capturado con `dataset_mode_v5`  
desde Webots. El modelo aprende a imitar la conducta humana para las tres fases de evasion:

| Fase | behavior_class | Descripcion |
|------|---------------|-------------|
| NORMAL    | 0 | Seguimiento de carril (izquierdo o derecho) |
| EVADING   | 1 | Cambio al carril derecho al detectar un vehiculo |
| RETURNING | 2 | Regreso al carril izquierdo una vez despejado |

**Diferencias respecto al notebook 26:**
- Dataset: `dataset_mode_v5` — columna `behavior_class` (antes `evasion_phase`)
- Ruta de imagenes: `behavioral_dataset_BC_<date>/images/{contributor_id}_frame_XXXXXX.jpg`
- Columnas extra en CSV ignoradas: `nav_command`, `lane_side`, `edge_detected`

**Salida del modelo:** angulo de direccion (radianes)  
**Export:** TorchScript `bc_evasion_model.pt` — listo para el controlador V1.15

## Paso 1 — Verificar GPU

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice activo:", DEVICE)

## Paso 2 — Cargar bibliotecas

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Bibliotecas cargadas.")

## Paso 3 — Rutas del proyecto

> **Actualizar `DATASET_NAME`** al nombre de la carpeta generada por `dataset_mode_v5`  
> cuando se grabo el dataset de evasion en el nuevo mundo.

In [ ]:
BASE_DIR     = r"D:\ML\Projects\Project_5_MR4010.10_Navegacion\MR4010.10_Navegacion"

# ── ACTUALIZAR AQUI cuando se grabe el nuevo dataset de evasion ──────────────
DATASET_NAME = "behavioral_dataset_BC_DDMMYYYY"   # <-- reemplazar con la carpeta real
# ─────────────────────────────────────────────────────────────────────────────

DATASET_DIR = os.path.join(BASE_DIR, "data", DATASET_NAME)
IMAGES_DIR  = os.path.join(DATASET_DIR, "images")
CSV_PATH    = os.path.join(DATASET_DIR, "measurements.csv")
MODELS_DIR  = os.path.join(BASE_DIR, "models")

MODEL_SCRIPT_PATH  = os.path.join(MODELS_DIR, "bc_evasion_model.pt")
MODEL_WEIGHTS_PATH = os.path.join(MODELS_DIR, "bc_evasion_weights.pt")
CONFIG_PATH        = os.path.join(MODELS_DIR, "bc_evasion_config.json")

os.makedirs(MODELS_DIR, exist_ok=True)

print("Dataset:", DATASET_DIR)
print("CSV:    ", CSV_PATH)
print("Images: ", IMAGES_DIR)
print("Model:  ", MODEL_SCRIPT_PATH)
print()
print("Dataset existe:", os.path.isdir(DATASET_DIR))
print("CSV existe:    ", os.path.isfile(CSV_PATH))

## Paso 4 — Cargar y explorar el dataset

In [ ]:
df = pd.read_csv(CSV_PATH)

print(f"Total de muestras: {len(df)}")
print(f"Columnas: {list(df.columns)}")
print()
df.head(5)

In [ ]:
print("Sesiones de grabacion:")
print(df.groupby('session_id').size().rename('frames'))
print()
print("Distribucion de behavior_class:")
print(df['behavior_class'].value_counts().sort_index())
print()
print("Modos (0=manual, 1=autonomo):")
print(df['autonomous_mode'].value_counts())
print()
print("Estadisticas de steering_angle:")
print(df['steering_angle'].describe())

## Paso 5 — Filtrar y limpiar el dataset

In [ ]:
# Solo muestras de conduccion manual (expert demonstrations)
df = df[df['autonomous_mode'] == 0].copy()
print(f"Muestras modo manual: {len(df)}")

# Verificar que las imagenes existen en disco
df['image_path'] = df['image_filename'].apply(lambda f: os.path.join(IMAGES_DIR, f))
mask_exists = df['image_path'].apply(os.path.exists)
missing = (~mask_exists).sum()
if missing > 0:
    print(f"ADVERTENCIA: {missing} imagenes no encontradas — eliminadas.")
df = df[mask_exists].copy()

# Limpiar nulos
df = df.dropna(subset=['steering_angle', 'speed_kmh', 'brake']).copy()
df['behavior_class']          = df['behavior_class'].astype(int)
df['lidar_obstacle_detected'] = df['lidar_obstacle_detected'].fillna(0).astype(int)
df['lidar_obstacle_distance'] = pd.to_numeric(
    df['lidar_obstacle_distance'], errors='coerce'
).fillna(0.0)

df = df.reset_index(drop=True)
print(f"Muestras limpias: {len(df)}  |  Sesiones: {df['session_id'].nunique()}")
print()
print("behavior_class despues de limpieza:")
print(df['behavior_class'].value_counts().sort_index())

## Paso 6 — Visualizar distribucion de etiquetas

In [ ]:
phase_labels = {0: 'NORMAL', 1: 'EVADING', 2: 'RETURNING'}
phase_colors = ['#4CAF50', '#F44336', '#FF9800']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribucion del Dataset BC Evasion (v5)', fontsize=14)

axes[0,0].hist(df['steering_angle'], bins=60, color='steelblue', edgecolor='white')
axes[0,0].set_title('Angulo de Direccion (rad)')
axes[0,0].axvline(0, color='red', linestyle='--', alpha=0.7)

axes[0,1].hist(df['speed_kmh'], bins=40, color='seagreen', edgecolor='white')
axes[0,1].set_title('Velocidad (km/h)')

axes[0,2].hist(df['brake'], bins=30, color='tomato', edgecolor='white')
axes[0,2].set_title('Freno (0-1)')

phase_counts = df['behavior_class'].value_counts().sort_index()
axes[1,0].bar(
    [phase_labels[p] for p in phase_counts.index],
    phase_counts.values,
    color=[phase_colors[p] for p in phase_counts.index]
)
axes[1,0].set_title('Distribucion de Fases')
for i, v in enumerate(phase_counts.values):
    axes[1,0].text(i, v + 1, str(v), ha='center', fontsize=10)

for phase, color, label in zip([0,1,2], phase_colors, ['NORMAL','EVADING','RETURNING']):
    subset = df[df['behavior_class'] == phase]['steering_angle']
    if len(subset) > 0:
        axes[1,1].hist(subset, bins=40, alpha=0.6, color=color, label=f'{label} (n={len(subset)})')
axes[1,1].set_title('Angulo de Direccion por Fase')
axes[1,1].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[1,1].legend(fontsize=8)

sess_counts = df['session_id'].value_counts()
axes[1,2].bar(range(len(sess_counts)), sess_counts.values, color='mediumpurple')
axes[1,2].set_title(f'Frames por Sesion ({len(sess_counts)} sesiones)')

plt.tight_layout()
plt.show()

print("Desbalance de fases:")
for p, label in phase_labels.items():
    n = (df['behavior_class'] == p).sum()
    print(f"  Fase {p} {label:10s}: {n:5d} muestras ({100*n/len(df):.1f}%)")

## Paso 7 — Visualizar imagenes de ejemplo por fase

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
fig.suptitle('Imagenes de ejemplo por fase de evasion', fontsize=13)

for row_idx, (phase, label) in enumerate([(0,'NORMAL'),(1,'EVADING'),(2,'RETURNING')]):
    subset = df[df['behavior_class'] == phase]
    n = min(4, len(subset))
    if n == 0:
        for col in range(4):
            axes[row_idx, col].axis('off')
        continue
    sample = subset.sample(n, random_state=SEED)
    for col, (_, r) in enumerate(sample.iterrows()):
        img = cv2.imread(r['image_path'])
        if img is None:
            axes[row_idx, col].axis('off')
            continue
        axes[row_idx, col].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[row_idx, col].set_title(f"{label}\nangle={r['steering_angle']:.3f}", fontsize=8)
        axes[row_idx, col].axis('off')
    for col in range(n, 4):
        axes[row_idx, col].axis('off')

plt.tight_layout()
plt.show()

## Paso 8 — Division por sesion (train / validacion)

> La division es **por sesion completa**, no por frame aleatorio.  
> Dividir por frame causa *data leakage*: frames consecutivos son casi identicos.  
> Si solo hay una sesion se usa split temporal 80/20.

In [ ]:
sessions = sorted(df['session_id'].unique())
print("Sesiones disponibles:", sessions)

val_session    = sessions[-1]
train_sessions = sessions[:-1]

if len(train_sessions) == 0:
    split    = int(len(df) * 0.8)
    df_train = df.iloc[:split].copy()
    df_val   = df.iloc[split:].copy()
    print("Solo una sesion — split temporal 80/20")
else:
    df_train = df[df['session_id'].isin(train_sessions)].copy()
    df_val   = df[df['session_id'] == val_session].copy()

df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(drop=True)

print(f"Train: {len(df_train)} muestras ({df_train['session_id'].nunique()} sesiones)")
print(f"Val:   {len(df_val)} muestras  (sesion: {val_session})")
print("\nFases en Train:"); print(df_train['behavior_class'].value_counts().sort_index())
print("\nFases en Val:");   print(df_val['behavior_class'].value_counts().sort_index())

## Paso 9 — Preprocesamiento de imagenes

**Recorte:** 38% superior (cielo) y 12% inferior (capo).  
**Resize:** 66x200 px (NVIDIA DAVE-2).  
**Normalizacion:** [0.0, 1.0] float32.

> Esta funcion debe ser **identica** en el controlador Webots y en este notebook.

In [ ]:
IMG_H       = 66
IMG_W       = 200
CROP_TOP    = 0.38
CROP_BOTTOM = 0.88


def preprocess_image(img_bgr, augment=False):
    """Preprocesa un frame BGR de Webots para el modelo BC."""
    h, w = img_bgr.shape[:2]
    y1 = int(h * CROP_TOP)
    y2 = int(h * CROP_BOTTOM)
    cropped = img_bgr[y1:y2, :]
    resized = cv2.resize(cropped, (IMG_W, IMG_H))
    img = resized.astype(np.float32) / 255.0

    if augment:
        img = np.clip(img + np.random.uniform(-0.3, 0.3), 0.0, 1.0)
        img = np.clip(img + np.random.normal(0, 0.02, img.shape).astype(np.float32),
                      0.0, 1.0)
    return img  # HWC float32


sample_img = cv2.imread(df['image_path'].iloc[0])
processed  = preprocess_image(sample_img)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Original {sample_img.shape[1]}x{sample_img.shape[0]}')
axes[0].axis('off')
axes[1].imshow(processed)
axes[1].set_title(f'Procesada {IMG_W}x{IMG_H}')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Paso 10 — Dataset PyTorch y DataLoader

In [ ]:
class BCDataset(Dataset):
    """
    Dataset de Behavioral Cloning.
    flip=True duplica cada muestra con volteo horizontal (angulo negado).
    """

    def __init__(self, df, augment=False, flip=False):
        self.df      = df.reset_index(drop=True)
        self.augment = augment
        self.n_base  = len(self.df)
        self.flip    = flip
        self.length  = self.n_base * 2 if flip else self.n_base

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        is_flipped = self.flip and (idx >= self.n_base)
        real_idx   = idx % self.n_base
        row        = self.df.iloc[real_idx]

        img = cv2.imread(row['image_path'])
        if img is None:
            img = np.zeros((320, 620, 3), dtype=np.uint8)

        img   = preprocess_image(img, augment=self.augment)
        angle = float(row['steering_angle'])

        if is_flipped:
            img   = img[:, ::-1, :].copy()
            angle = -angle

        img_tensor = torch.from_numpy(img).permute(2, 0, 1)
        return img_tensor, torch.tensor(angle, dtype=torch.float32)


test_ds = BCDataset(df_train.head(10), augment=False, flip=True)
img_t, ang_t = test_ds[0]
print(f"Tensor imagen: {img_t.shape}  dtype={img_t.dtype}")
print(f"Angulo:        {ang_t.item():.4f} rad")
print(f"Dataset size (con flip): {len(test_ds)}")

## Paso 11 — Pesos de fase y DataLoaders

`WeightedRandomSampler` sobremuestra automaticamente EVADING y RETURNING  
para que el modelo no ignore la maniobra de evasion.  

> Ajustar `PHASE_WEIGHTS` si alguna fase queda subrepresentada en el nuevo dataset.

In [ ]:
BATCH_SIZE    = 32
NUM_WORKERS   = 0
PHASE_WEIGHTS = {0: 1.0, 1: 5.0, 2: 4.0}

train_dataset = BCDataset(df_train, augment=True,  flip=True)
val_dataset   = BCDataset(df_val,   augment=False, flip=False)

base_weights   = df_train['behavior_class'].map(PHASE_WEIGHTS).values.tolist()
sample_weights = base_weights + base_weights
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    sampler=sampler, num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)

print(f"Train batches: {len(train_loader)}  ({len(train_dataset)} muestras con flip)")
print(f"Val batches:   {len(val_loader)}  ({len(val_dataset)} muestras)")
print("\nPesos de fase:")
for p, w in PHASE_WEIGHTS.items():
    n = (df_train['behavior_class'] == p).sum()
    print(f"  Fase {p} {phase_labels[p]:10s}: peso={w:.1f}  n={n}")

## Paso 12 — Arquitectura del modelo CNN (NVIDIA DAVE-2)

```
Input (3, 66, 200)  CHW
  Conv2d(3  -> 24, 5x5, s=2) + ELU  -> (24, 31, 98)
  Conv2d(24 -> 36, 5x5, s=2) + ELU  -> (36, 14, 47)
  Conv2d(36 -> 48, 5x5, s=2) + ELU  -> (48,  5, 22)
  Conv2d(48 -> 64, 3x3, s=1) + ELU  -> (64,  3, 20)
  Conv2d(64 -> 64, 3x3, s=1) + ELU  -> (64,  1, 18)
  Flatten -> 1152
  Linear(1152->100) + ELU + Dropout(0.3)
  Linear(100 -> 50) + ELU + Dropout(0.2)
  Linear( 50 -> 10) + ELU
  Linear( 10 ->  1)  -> steering_angle
```

In [ ]:
class DAVE2(nn.Module):
    """NVIDIA DAVE-2 para Behavioral Cloning en Webots. Input CHW (3, 66, 200)."""

    def __init__(self, dropout_1=0.3, dropout_2=0.2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,  24, kernel_size=5, stride=2), nn.ELU(),
            nn.Conv2d(24, 36, kernel_size=5, stride=2), nn.ELU(),
            nn.Conv2d(36, 48, kernel_size=5, stride=2), nn.ELU(),
            nn.Conv2d(48, 64, kernel_size=3, stride=1), nn.ELU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ELU(),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1152, 100), nn.ELU(), nn.Dropout(dropout_1),
            nn.Linear(100,  50),  nn.ELU(), nn.Dropout(dropout_2),
            nn.Linear(50,   10),  nn.ELU(),
            nn.Linear(10,    1)
        )

    def forward(self, x):
        return self.regressor(self.features(x)).squeeze(1)


model = DAVE2().to(DEVICE)

with torch.no_grad():
    dummy = torch.zeros(1, 3, IMG_H, IMG_W).to(DEVICE)
    out   = model(dummy)
    print("Input shape: ", dummy.shape)
    print("Output shape:", out.shape)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parametros totales: {total_params:,}")

## Paso 13 — Resumen del modelo

In [ ]:
print(model)
print()
print(f"{'Layer':<35} {'Params':>10}")
print("-" * 47)
for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        params = sum(p.numel() for p in module.parameters())
        print(f"{name:<35} {params:>10,}")

## Paso 14 — Configurar entrenamiento

In [ ]:
EPOCHS      = 40
LR          = 1e-4
PATIENCE    = 8
LR_PATIENCE = 4

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=LR_PATIENCE
)
scaler = GradScaler('cuda', enabled=torch.cuda.is_available())

print(f"Optimizer : Adam  lr={LR}")
print(f"Loss      : MSELoss")
print(f"AMP (fp16): {torch.cuda.is_available()}")
print(f"Early stop: patience={PATIENCE} epocas")

## Paso 15 — Entrenar el modelo

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}

best_val_loss    = float('inf')
best_epoch       = 0
patience_counter = 0

for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = 0.0
    train_mae  = 0.0

    for imgs, angles in train_loader:
        imgs   = imgs.to(DEVICE)
        angles = angles.to(DEVICE)

        optimizer.zero_grad()
        with autocast('cuda', enabled=torch.cuda.is_available()):
            preds = model(imgs)
            loss  = criterion(preds, angles)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_mae  += torch.abs(preds.detach() - angles).mean().item()

    train_loss /= len(train_loader)
    train_mae  /= len(train_loader)

    model.eval()
    val_loss = 0.0
    val_mae  = 0.0

    with torch.no_grad():
        for imgs, angles in val_loader:
            imgs   = imgs.to(DEVICE)
            angles = angles.to(DEVICE)
            with autocast('cuda', enabled=torch.cuda.is_available()):
                preds = model(imgs)
                loss  = criterion(preds, angles)
            val_loss += loss.item()
            val_mae  += torch.abs(preds - angles).mean().item()

    val_loss /= len(val_loader)
    val_mae  /= len(val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_mae'].append(train_mae)
    history['val_mae'].append(val_mae)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_epoch       = epoch
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_WEIGHTS_PATH)
        tag = '* BEST'
    else:
        patience_counter += 1
        tag = f'({patience_counter}/{PATIENCE})'

    lr_now = optimizer.param_groups[0]['lr']
    print(f"Ep {epoch:03d}/{EPOCHS}  "
          f"loss={train_loss:.5f}  val_loss={val_loss:.5f}  "
          f"mae={train_mae:.5f}  val_mae={val_mae:.5f}  "
          f"lr={lr_now:.2e}  {tag}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping en epoca {epoch}. Mejor: epoca {best_epoch}.")
        break

model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))
print(f"\nMejor modelo restaurado (epoca {best_epoch}, val_loss={best_val_loss:.6f})")

## Paso 16 — Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history['train_loss'], label='Train Loss (MSE)')
axes[0].plot(history['val_loss'],   label='Val Loss (MSE)')
axes[0].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5,
                label=f'Best ep {best_epoch}')
axes[0].set_title('Evolucion del Error (MSE)')
axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history['train_mae'], label='Train MAE')
axes[1].plot(history['val_mae'],   label='Val MAE')
axes[1].axvline(best_epoch - 1, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Evolucion del MAE')
axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('MAE (rad)')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"Mejor epoca: {best_epoch}")
print(f"Mejor val_loss (MSE): {best_val_loss:.6f}")
print(f"Mejor val_mae:        {history['val_mae'][best_epoch-1]:.6f} rad")

## Paso 17 — Evaluar el modelo

In [ ]:
model.eval()
y_true_all = []
y_pred_all = []
phases_all = []

with torch.no_grad():
    for _, row in df_val.iterrows():
        img = cv2.imread(row['image_path'])
        if img is None:
            continue
        img_proc = preprocess_image(img, augment=False)
        t = torch.from_numpy(img_proc).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
        pred = model(t).item()
        y_true_all.append(float(row['steering_angle']))
        y_pred_all.append(float(pred))
        phases_all.append(int(row['behavior_class']))

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)
phases_all = np.array(phases_all)

mae  = mean_absolute_error(y_true_all, y_pred_all)
rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))

print(f"{'='*50}")
print(f"  Evaluacion global (validacion)")
print(f"  MAE  = {mae:.5f} rad  (~{np.degrees(mae):.2f} grados)")
print(f"  RMSE = {rmse:.5f} rad")
print(f"{'='*50}")

print("\nError por fase:")
for phase, label in phase_labels.items():
    mask = phases_all == phase
    if mask.sum() == 0:
        print(f"  Fase {phase} {label:10s}: sin muestras")
        continue
    p_mae  = mean_absolute_error(y_true_all[mask], y_pred_all[mask])
    p_rmse = np.sqrt(mean_squared_error(y_true_all[mask], y_pred_all[mask]))
    print(f"  Fase {phase} {label:10s}: MAE={p_mae:.5f}  RMSE={p_rmse:.5f}  n={mask.sum()}")

## Paso 18 — Scatter: predicciones vs reales

In [ ]:
phase_colors_map = {0: '#4CAF50', 1: '#F44336', 2: '#FF9800'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for phase, label in phase_labels.items():
    mask = phases_all == phase
    if mask.sum() == 0:
        continue
    axes[0].scatter(
        y_true_all[mask], y_pred_all[mask],
        c=phase_colors_map[phase], label=label, alpha=0.6, s=18
    )

lim = max(abs(y_true_all).max(), abs(y_pred_all).max()) * 1.1
axes[0].plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='Prediccion perfecta')
axes[0].set_xlabel('Angulo real (rad)')
axes[0].set_ylabel('Angulo predicho (rad)')
axes[0].set_title('Predicciones vs Reales por Fase')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

abs_error = np.abs(y_true_all - y_pred_all)
axes[1].plot(abs_error, alpha=0.5, color='steelblue', linewidth=0.8)
axes[1].axhline(mae, color='red', linestyle='--', label=f'MAE={mae:.4f} rad')
for phase, color in phase_colors_map.items():
    mask = phases_all == phase
    if mask.sum() > 0:
        axes[1].scatter(np.where(mask)[0], abs_error[mask],
                        c=color, s=8, alpha=0.5, label=phase_labels[phase])
axes[1].set_xlabel('Frame (validacion)')
axes[1].set_ylabel('Error absoluto (rad)')
axes[1].set_title('Error Absoluto por Frame')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Paso 19 — Visualizar predicciones sobre imagenes reales

In [ ]:
def draw_steering_overlay(img_rgb, true_angle, pred_angle):
    h, w = img_rgb.shape[:2]
    cx, cy, L = w // 2, h - 5, 25
    out = img_rgb.copy()
    tx = int(cx + L * np.sin(true_angle * 3))
    ty = int(cy - L * np.cos(true_angle * 3))
    cv2.arrowedLine(out, (cx, cy), (tx, ty), (0, 220, 0), 2, tipLength=0.4)
    px = int(cx + L * np.sin(pred_angle * 3))
    py = int(cy - L * np.cos(pred_angle * 3))
    cv2.arrowedLine(out, (cx, cy), (px, py), (220, 50, 50), 2, tipLength=0.4)
    return out


sample_rows = []
for phase in [0, 1, 2]:
    subset = df_val[df_val['behavior_class'] == phase]
    n = min(4, len(subset))
    if n > 0:
        sample_rows.append(subset.sample(n, random_state=SEED))

sample_df = pd.concat(sample_rows).reset_index(drop=True)

model.eval()
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
fig.suptitle('Predicciones — Verde: real  Rojo: predicho', fontsize=12)

for i, (_, row) in enumerate(sample_df.iterrows()):
    ax = axes[i // 4, i % 4]
    img = cv2.imread(row['image_path'])
    if img is None:
        ax.axis('off')
        continue
    img_proc = preprocess_image(img, augment=False)
    t = torch.from_numpy(img_proc).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(t).item()
    true = float(row['steering_angle'])
    vis  = draw_steering_overlay((img_proc * 255).astype(np.uint8), true, pred)
    ax.imshow(vis)
    ax.set_title(
        f"{phase_labels[int(row['behavior_class'])]}\nT:{true:.3f} P:{pred:.3f}",
        fontsize=7
    )
    ax.axis('off')

for j in range(len(sample_df), 12):
    axes[j // 4, j % 4].axis('off')

plt.tight_layout()
plt.show()

## Paso 20 — Guardar modelo y configuracion

- **`bc_evasion_weights.pt`** — state dict (requiere la clase `DAVE2` para cargar)
- **`bc_evasion_model.pt`** — TorchScript traced (no requiere la clase, listo para el controlador V1.15)

In [ ]:
print("Weights guardados en:", MODEL_WEIGHTS_PATH)

model.eval()
dummy_input = torch.zeros(1, 3, IMG_H, IMG_W).to(DEVICE)
traced = torch.jit.trace(model, dummy_input)
torch.jit.save(traced, MODEL_SCRIPT_PATH)
print("TorchScript guardado en:", MODEL_SCRIPT_PATH)

config = {
    "img_h":          IMG_H,
    "img_w":          IMG_W,
    "crop_top":       CROP_TOP,
    "crop_bottom":    CROP_BOTTOM,
    "model_path":     MODEL_SCRIPT_PATH,
    "dataset_name":   DATASET_NAME,
    "val_mae_rad":    float(mae),
    "val_rmse_rad":   float(rmse),
    "best_epoch":     best_epoch,
    "train_samples":  len(df_train),
    "val_samples":    len(df_val),
    "phase_weights":  PHASE_WEIGHTS,
    "notebook":       "29 - BC_Evasion_v5"
}
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=2)
print("Config guardado en:", CONFIG_PATH)
print()
print(json.dumps(config, indent=2))

## Paso 21 — Verificar carga e inferencia del modelo exportado

In [ ]:
loaded_model = torch.jit.load(MODEL_SCRIPT_PATH, map_location=DEVICE)
loaded_model.eval()
print("Modelo TorchScript cargado desde:", MODEL_SCRIPT_PATH)

test_row  = df_val.sample(1, random_state=7).iloc[0]
test_img  = cv2.imread(test_row['image_path'])
test_proc = preprocess_image(test_img, augment=False)
test_t    = torch.from_numpy(test_proc).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

with torch.no_grad():
    test_pred = loaded_model(test_t).item()

print(f"Fase: {test_row['behavior_class']}  "
      f"Real: {test_row['steering_angle']:.4f} rad  "
      f"Prediccion: {test_pred:.4f} rad")

plt.figure(figsize=(6, 2.5))
plt.imshow(test_proc)
plt.title(
    f"Fase {test_row['behavior_class']}  |  "
    f"Real={test_row['steering_angle']:.4f}  Pred={test_pred:.4f}"
)
plt.axis('off')
plt.show()

---
## Criterios de aceptacion y notas

### Criterios de aceptacion
- **MAE < 0.05 rad** (~3 grados) para considerar el modelo apto para el controlador.
- Error en Fase 1 (EVADING) debe ser menor o comparable al de Fase 0 (NORMAL).

### Si el error es alto
- Grabar mas sesiones de evasion — objetivo: **>= 300 frames de fase 1 y 2** por sesion.
- Aumentar `PHASE_WEIGHTS[1]` y `PHASE_WEIGHTS[2]` si EVADING/RETURNING siguen con error alto.
- Verificar que las fases 1 y 2 esten bien representadas en el debug del controlador antes de grabar.

### Recordatorio de grabacion
- Grabar en **modo manual** (`autonomous_mode=0`) con dataset mode activo (tecla D).
- Tener un vehiculo en la escena para que `behavior_class` cambie a 1 y 2.
- El debounce de 5 frames (`VEHICLE_CONFIRM_FRAMES=5`) requiere que el vehiculo sea visible
  por al menos 5 frames consecutivos antes de que `behavior_class` cambie a 1.
- Grabar al menos 2 sesiones separadas para tener split por sesion en train/val.